In [1]:
%matplotlib tk
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from torch.nn.functional import conv2d, conv3d
from scipy.ndimage import convolve, generate_binary_structure
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Traceback (most recent call last):
  File "C:\Users\hasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\matplotlib\cbook\__init__.py", line 304, in process
    func(*args, **kwargs)
  File "C:\Users\hasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\matplotlib\animation.py", line 904, in _start
    self._init_draw()
  File "C:\Users\hasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\matplotlib\animation.py", line 1748, in _init_draw
    self._draw_frame(frame_data)
  File "C:\Users\hasan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\matplotlib\animation.py", line 1767, in _draw_frame
    self._drawn_artists = self._func(framedata, *self._args)
                          ^^^^^^^^

In [36]:
def wave_equation_2d(psi_0, psi_1, dt, dx, dy, c, mask=None):
    """
    Simulates the 2D wave equation using explicit finite differences with Dirichlet (zero) boundary conditions inside a mask.
    mask: boolean tensor, True inside the domain, False outside
    """
    c2 = c**2
    NX, NY = psi_0.shape
    psi_2 = torch.zeros_like(psi_0)
    # Interior points (finite difference Laplacian)
    psi_2[1:-1,1:-1] = (
        2 * psi_1[1:-1,1:-1] - psi_0[1:-1,1:-1]
        + c2 * dt**2 * (
            (psi_1[2:,1:-1] - 2*psi_1[1:-1,1:-1] + psi_1[:-2,1:-1]) / dx**2
            + (psi_1[1:-1,2:] - 2*psi_1[1:-1,1:-1] + psi_1[1:-1,:-2]) / dy**2
        )
    )
    if mask is not None:
        # Dirichlet boundary: set all points outside mask to zero
        psi_2[~mask] = 0
        # Set boundary points (where mask is True but at least one neighbor is False) to zero
        # Top
        psi_2[0, mask[0,:]] = 0
        # Bottom
        psi_2[-1, mask[-1,:]] = 0
        # Left
        psi_2[mask[:,0], 0] = 0
        # Right
        psi_2[mask[:,-1], -1] = 0
    else:
        # Standard Dirichlet boundaries (whole grid)
        psi_2[0, :] = 0
        psi_2[-1, :] = 0
        psi_2[:, 0] = 0
        psi_2[:, -1] = 0
    return psi_2

In [1]:
# Efficient animation: update psi in-place and avoid storing all iterations

NX, NY = 500, 500
n_iterations = 1000
dx = 0.1
dy = 0.1
c = 1.0
courant_number = 0.2
dt = courant_number * min(dx, dy)**2 / c
x, y = torch.meshgrid(torch.linspace(0, NX-1, NX), torch.linspace(0, NY-1, NY), indexing='ij')

# Ellipse parameters
center_x, center_y = NX // 2, NY // 2
ellipse_a = NX // 2.5  # semi-major axis (x-direction)
ellipse_b = NY // 3.5  # semi-minor axis (y-direction)
ellipse_mask = (((x - center_x) / ellipse_a) ** 2 + ((y - center_y) / ellipse_b) ** 2) <= 1
ellipse_mask = torch.tensor(ellipse_mask, dtype=torch.bool, device=device)

# Calculate left focus of the ellipse
c_ellipse = np.sqrt(np.abs(ellipse_a**2 - ellipse_b**2))
left_focus_x = center_x - c_ellipse
left_focus_y = center_y

# Place initial wave at the left focus
psi_0 = 8*torch.exp(-((x - left_focus_x)**2 + (y - left_focus_y)**2) / (2 * (NX/40)**2)).to(device)
psi_1 = psi_0.clone()

# ellipse_mask is a boolean tensor with True inside the ellipse
plt.imshow(ellipse_mask.cpu().numpy().T)
plt.title('Ellipse Mask')

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_facecolor('white')
im = ax.imshow(psi_0.cpu().numpy().T, cmap='seismic', extent=(0, NX*dx, 0, NY*dy), vmax=8, vmin=-8)
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)
t = np.linspace(0, n_iterations * dt, n_iterations + 1)

# Only keep two arrays in memory, update in-place
psi_prev = psi_0.clone()
psi_curr = psi_1.clone()
import gc

source_amplitude = 0.00#01  # Amplitude of the source term
psi_source = source_amplitude*torch.exp(-((x - left_focus_x)**2 + (y - left_focus_y)**2) / (2 * (NX/100)**2)).to(device)

frames_to_advance = 50  # Number of simulation steps per animation frame

def update(frame):
    global psi_prev, psi_curr
    for i in range(frames_to_advance):
        t_global = (frame * frames_to_advance + i) * dt
        source_term = psi_source * np.sin(2 * np.pi * t_global / (dt * frames_to_advance * 50))
        psi_next = wave_equation_2d(psi_prev, psi_curr, dt, dx, dy, c, mask=ellipse_mask) + source_term
        psi_prev = psi_curr.clone()
        psi_curr = psi_next.clone()
    im.set_array(psi_curr.cpu().numpy().T)
    time_text.set_text(f'Time = {frame*frames_to_advance*dt:.2f} s')
    return [im, time_text]

ani = animation.FuncAnimation(fig, update, frames=n_iterations, interval=5, blit=True)
plt.colorbar(im, ax=ax, label='Amplitude')
plt.xlabel('X')
plt.ylabel('Y')
plt.show()
#ani.save('/Users/hasan/Python Animations/wave_equation_animation_ellipse.gif', writer='pillow', fps=50,dpi=200)

NameError: name 'torch' is not defined